# Lab 4: Pairwise Sequence Alignment and Database Search

Compare global and local alignment, scoring schemes, and biological interpretation using Biopython. Use BLAST commands in command line. 

**Goals:** 
- choose an alignment strategy, inspect scores and gaps, and explain how scoring parameters affect conclusions.
- get familar with using command lines for one of the most fundamental bioinformatics tasks, finding similar sequences. 

---
## Part 0 — Environment Setup
Hopefully by now you know how to activate the Bio2 environment in your VS Code. 

Refer Lab 03 if you need help.

In [108]:
# Run once on Colab or a fresh environment — skip if bio2 is already active
# If you are running this notebook on Colab or your own laptop, you may need install the required packages first.
# If so, uncomment the following line to install the required packages.
# %pip install -q biopython numpy matplotlib pandas requests

## Check if the bio2 environment is setup properly

In [109]:
from io import StringIO
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests

from Bio import Align, Entrez, SeqIO
from Bio.Align import substitution_matrices
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction

print("All imports successful ✓")

All imports successful ✓


### Proceed if all imports sucessfully. 

## Part 1 — Pairwise Protein Alignment

### 1a. Global alignment — compare two matrices

In [110]:
seq1 = Seq("MAEGEITTFTALTEKFNLPPGNYKKPKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQL"
           "SAESVGEVYIKSTETGQYLAMDTSGLLYGSQTPSEECLFLERLEENHYNTYTSKKHAEKNW"
           "FVGLKKNGSCKRGPRTHYGQKAILFLPLPV")
seq2 = Seq("MAEGNITAFTALTEKFNLPPGNYKKAKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQL"
           "SAESVGEVYIKSTETGQYLAMDTSGLLYGSQTPTEECLFLERLEENHYNTYTSKKHAEKNW"
           "FVGLKKNGSCKRGPRTHYGQKAILFLPLPV")

for matrix_name in ["PAM250", "BLOSUM62"]:
    matrix  = substitution_matrices.load(matrix_name)
    aligner = Align.PairwiseAligner()
    aligner.substitution_matrix = matrix
    aligner.open_gap_score      = -10
    aligner.extend_gap_score    = -0.5
    aligner.mode                = "global"
    best = next(aligner.align(seq1, seq2))
    print(f"\n{'='*50}\nMatrix: {matrix_name}   Score: {best.score:.2f}\n{'='*50}")
    print(best)


Matrix: PAM250   Score: 762.00
target            0 MAEGEITTFTALTEKFNLPPGNYKKPKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQ
                  0 ||||.||.|||||||||||||||||.||||||||||||||||||||||||||||||||||
query             0 MAEGNITAFTALTEKFNLPPGNYKKAKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQ

target           60 LSAESVGEVYIKSTETGQYLAMDTSGLLYGSQTPSEECLFLERLEENHYNTYTSKKHAEK
                 60 ||||||||||||||||||||||||||||||||||.|||||||||||||||||||||||||
query            60 LSAESVGEVYIKSTETGQYLAMDTSGLLYGSQTPTEECLFLERLEENHYNTYTSKKHAEK

target          120 NWFVGLKKNGSCKRGPRTHYGQKAILFLPLPV 152
                120 |||||||||||||||||||||||||||||||| 152
query           120 NWFVGLKKNGSCKRGPRTHYGQKAILFLPLPV 152


Matrix: BLOSUM62   Score: 792.00
target            0 MAEGEITTFTALTEKFNLPPGNYKKPKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQ
                  0 ||||.||.|||||||||||||||||.||||||||||||||||||||||||||||||||||
query             0 MAEGNITAFTALTEKFNLPPGNYKKAKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQ

target           60 LSAESVGEVY

> **Questions 1a:1–2**
>
> 1a.1. Do both matrices produce the same alignment? Where do they differ if not?
> 1a.2. Which matrix gives a higher score for these nearly identical sequences? Is that expected?

**Your answers:**

1a.1 

1a.2

### 1b. Same as 1a, but using two different sequences. 
``` 
seq1 = Seq("MAEGCDEITTFTALTEKFNLPPGYKKLLYCSNGGHFLRI")
seq2 = Seq("MACDEGEITTFTALTEKFNLPPGNYWKKPKLLYCSNGGHFLRI")
``` 
Use any approach that works for you: copy & paste (and revise), type, or use generative AI. 

In [111]:
# your work here

> **Questions 1b:1–2**
>
> 1b.1. Do both matrices produce the same alignment? Where do they differ if not?

> 1b.2. Do you like how you implemented 1b?

**Your answers:**

1b.1 

1b.2

### 1c. Local alignment to find a conserved domain

In [112]:
query  = Seq("HEAGAWGHEE")
target = Seq("PAWHEAE")

for matrix_name in ["PAM250", "BLOSUM50", "BLOSUM62"]:
    matrix  = substitution_matrices.load(matrix_name)
    aligner = Align.PairwiseAligner()
    aligner.substitution_matrix = matrix
    aligner.open_gap_score      = -8
    aligner.extend_gap_score    = -0.5
    aligner.mode                = "local"
    best = next(aligner.align(query, target))
    print(f"\n=== {matrix_name} | Score: {best.score:.1f} ===")
    print(best)


=== PAM250 | Score: 22.0 ===
target            4 AWGHEE 10
                  0 ||...|  6
query             1 AWHEAE  7


=== BLOSUM50 | Score: 28.0 ===
target            4 AWGHE 9
                  0 ||-|| 5
query             1 AW-HE 5


=== BLOSUM62 | Score: 20.0 ===
target            4 AWGHE 9
                  0 ||-|| 5
query             1 AW-HE 5



> **Questions 1c:1–2**
>
> 1c.1. Does the best local alignment change between matrices?
>
> 1c.2. Adjust the gap-open penalty from `-8` to `-4`. How does the alignment change?

**Your answers:**

1c.1. 

1c.2. 

---
## Part 2 — PAM vs BLOSUM: Which Matrix to Use?

### 2a. Effect of evolutionary distance on alignment score

In [113]:
seq1   = Seq("ACDEFGHIKLMNPQRSTVWY")
seq2   = Seq("ACDEFGHIKLMNPQRSTVWY")
seq3 = Seq("WYVSRQPNMLKIHGFEDCA")

matrices = {
    "PAM30":    substitution_matrices.load("PAM30"),
    "PAM250":   substitution_matrices.load("PAM250"),
    "BLOSUM90": substitution_matrices.load("BLOSUM90"),
    "BLOSUM62": substitution_matrices.load("BLOSUM62")
}

print(f"{'Matrix':<12}  {'seq1-seq2':>12}  {'seq1-seq3':>12}  {'Difference':>12}")
print("-" * 52)
for name, mat in matrices.items():
    aligner = Align.PairwiseAligner()
    aligner.substitution_matrix = mat
    aligner.mode = "global"
    sc = aligner.align(seq1, seq2).score
    sd = aligner.align(seq1, seq3).score
    print(f"{name:<12}  {sc:>12.1f}  {sd:>12.1f}  {sc-sd:>12.1f}")

Matrix           seq1-seq2     seq1-seq3    Difference
----------------------------------------------------
PAM30                164.0          -6.0         170.0
PAM250               118.0          15.0         103.0
BLOSUM90             134.0           3.0         131.0
BLOSUM62             116.0           4.0         112.0


> **Questions 2a:1–3**
>
> 2a.1. What can you tell about the relationship between seq1, seq2, and seq3 by looking at their sequences?
>
> 2a.2. Do the alignment results support your observation?
>
> 2a.3. What conclusion can you draw by checking the differences?

**Your answers:**

2a.1. 

2a.2.

2a.3. 

### 2b: same as 2a, but create and use your own sequences (seq1, seq2, seq3).

This is a very important skill that is good to have. Create your own test cases, for which you know if the results are correct or make sense.

In [114]:
# complete 2b here


## 2c — If you need more practice, aligning real protein sequences from FASTA shown below

Hemoglobin beta subunit from **human** and **horse** — two mammals whose hemoglobins diverged ~80 million years ago.

In [115]:
fasta_human = """>sp|P68871|HBB_HUMAN Hemoglobin subunit beta OS=Homo sapiens
MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLG
AFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVAN
ALAHKYH
"""

fasta_horse = """>sp|P02062|HBB_HORSE Hemoglobin subunit beta OS=Equus caballus
MVHLSEEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSNPAAVMGNPKVKAHGKKVLH
SFSDGLKHLDNLKGTFAALSELHCDKLHVDPENFRLLGNVLVVVLARHFGKDFTPELQASYQKVVAGVAN
ALAHKYH
"""



### More on substution matrices: decision reference

| Situation | Recommended matrix |
|---|---|
| Closely related (> 85% identity) | BLOSUM80 or BLOSUM90 |
| Moderately related (30–85%) | **BLOSUM62** (default) |
| Distantly related (< 30%) | BLOSUM30, BLOSUM45, or PAM250 |
| Short peptide / domain search | PAM30 or BLOSUM80 |

> **Questions **
>
> Which matrix gives the largest score difference between identical and scrambled sequences? Is that useful?
> For a database search where you don't know the evolutionary distance, which matrix would you choose and why?

## Part 3 -- Database search using BLAST+ 


All the exercises for database search will be done using command lines.

BLAST+ includes makeblastdb, blastp, blastn, blastx, tblastn, tblastx, and utilities such as blastdbcmd.

You need BLAST+ installed to use its commands. 

If you have active Bio2 environment created for this class, BLAST+ is ready for you to use.  

NOTE: You need to activate Bio2 environment separately in terminal so you can use command lines.

See [NCBI BLAST manual](https://www.ncbi.nlm.nih.gov/books/NBK279690/)

Below show some simple usages using datasets prepared for this class.

Databases available under /nobackup/yye/Bio2/datasets/dbsearch (on silo):
- Ecoli_protein.faa -- protein sequences from E.coli K12
- Ecoli_genomic.fna -- gene sequences from E.coli K12 
- query_protein.faa -- query proteins from a different genome
- query_genomic.fna -- query genomic sequences from a different genome

Copy these four files to your own folder (you cannot write to the /nobackup/yye/..). Then do the following.

- Step 1: make database using makeblastdb. 
- Step 2: search query against database using the correct blast commands (blastp for proteins against proteins, blastn for DNA against DNA).

Some notes:
- Make sure that you inspect the files created by makeblastdb before you dive into the next step; you can also use blastdbcmd command to show the information of a search database. 
- You can look up usage online for each of the commands; you may also simply call the command without providing any arguments to show the usage. 

> **Questions 3:1–2**
>
> 3.1. How many files does makeblastdb create from a protein database? A genomic database?
>
> 3.2. Do blastp and blastn reveal similar sequences for the queries?